# Task 103: ehtim_imaging — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Radio interferometry imaging using CLEAN algorithm

**Paper**: Interferometric Imaging Directly with Closure Phases and Closure Amplitudes
**Repository**: https://github.com/achael/eht-imaging

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 21.29 dB ← 🔧 修复前: 19.18 dB (Round 1 修复前: 12.98 dB)
- **SSIM**: 0.4990
- **Evaluation**: VLBI sparse aperture imaging (CLEAN algorithm)

### Mathematical Formulation

**Digital holography** records interference between reference $R$ and object $O$ waves:

$$I(x,y) = |R+O|^2 = |R|^2 + |O|^2 + R^*O + RO^*$$

**Numerical reconstruction** via Fresnel back-propagation:
$$U(x',y',d) = \mathcal{F}^{-1}\left\{\mathcal{F}\{I \cdot R^*\} \cdot H(f_x,f_y,d)\right\}$$

where $H = \exp\left(i\frac{2\pi d}{\lambda}\sqrt{1-\lambda^2 f_x^2 - \lambda^2 f_y^2}\right)$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
ehtim_imaging - Event Horizon Telescope VLBI Image Reconstruction
==================================================================
From sparse VLBI visibility measurements (complex Fourier components at specific
uv-points), reconstruct a radio image of a black hole shadow.

Physics:
  - Forward: Visibilities V(u,v) = FFT[I(x,y)] sampled at sparse uv-points
  - Simulate uv-coverage from Earth-rotation synthesis with ~8 stations
  - Inverse: CLEAN algorithm (dirty image + iterative deconvolution)
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`grid_visibilities`, `make_dirty_image_fft`, `make_dirty_beam_fft`, `clean_algorithm`, `main`

In [ ]:
def grid_visibilities(vis, u, v, N, fov):
    """Grid sparse visibilities onto regular UV grid."""
    du = 1.0 / fov
    uv_grid = np.zeros((N, N), dtype=complex)
    weight_grid = np.zeros((N, N))
    for k in range(len(u)):
        iu = int(np.round(u[k] / du)) + N // 2
        iv = int(np.round(v[k] / du)) + N // 2
        if 0 <= iu < N and 0 <= iv < N:
            uv_grid[iv, iu] += vis[k]
            weight_grid[iv, iu] += 1.0
    mask = weight_grid > 0
    uv_grid[mask] /= weight_grid[mask]
    return uv_grid, weight_grid

def make_dirty_image_fft(vis, u, v, N, fov):
    """Compute dirty image via FFT gridding."""
    uv_grid, _ = grid_visibilities(vis, u, v, N, fov)
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(uv_grid))).real

def make_dirty_beam_fft(u, v, N, fov):
    """Compute dirty beam via FFT gridding."""
    ones = np.ones(len(u), dtype=complex)
    uv_grid, _ = grid_visibilities(ones, u, v, N, fov)
    beam = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(uv_grid))).real
    if beam.max() > 0:
        beam /= beam.max()
    return beam

def clean_algorithm(dirty_image, dirty_beam, gain=0.1, niter=2000, threshold=0.01,
                    restore_sigma=None):
    """Hogbom CLEAN algorithm — fast slicing implementation.

    Parameters
    ----------
    restore_sigma : float or None
        Sigma (in pixels) for the Gaussian restoring beam.  When *None* the
        sigma is derived from the dirty-beam FWHM (original behaviour).
        A value of ~1.5 px for this 64-pixel / 200 µas FOV gives a good
        PSNR-SSIM trade-off.
    """
    N = dirty_image.shape[0]
    residual = dirty_image.copy()
    components = np.zeros_like(dirty_image)
    peak_val = np.abs(residual).max()
    thresh = threshold * peak_val
    bc = N // 2

    for it in range(niter):
        peak_idx = np.unravel_index(np.argmax(np.abs(residual)), residual.shape)
        peak = residual[peak_idx]
        if np.abs(peak) < thresh:
            break
        components[peak_idx] += gain * peak
        sy = peak_idx[0] - bc
        sx = peak_idx[1] - bc
        y1r = max(0, sy); y2r = min(N, N+sy)
        x1r = max(0, sx); x2r = min(N, N+sx)
        y1b = max(0, -sy); y2b = min(N, N-sy)
        x1b = max(0, -sx); x2b = min(N, N-sx)
        residual[y1r:y2r, x1r:x2r] -= gain * peak * dirty_beam[y1b:y2b, x1b:x2b]

    # Restore — use caller-supplied sigma or fall back to dirty-beam FWHM
    if restore_sigma is None:
        profile = dirty_beam[bc, :]
        profile = profile / (profile.max() + 1e-15)
        hm = np.where(profile >= 0.5)[0]
        fwhm = max(hm[-1] - hm[0], 2) if len(hm) > 1 else 3
        sigma = fwhm / 2.35
    else:
        sigma = restore_sigma
    xx = np.arange(N) - N/2
    XX, YY = np.meshgrid(xx, xx)
    clean_beam = np.exp(-0.5*(XX**2+YY**2)/sigma**2)
    # Peak-normalize (not sum-normalize) to preserve flux scale
    clean_beam /= clean_beam.max()
    restored = fftconvolve(components, clean_beam, mode='same') + residual
    return restored, components, residual, it+1

def main():
    print("="*60)
    print("Task 103: EHT VLBI Image Reconstruction")
    print("="*60)
    print("[1] Generating crescent black hole shadow model ...")
    gt_image = create_crescent_image(N_PIX, FOV_UAS)
    print(f"    Image: {N_PIX}x{N_PIX}, FOV={FOV_UAS} μas")
    print("[2] Generating uv-coverage ...")
    u, v = generate_uv_coverage(N_STATIONS, OBS_HOURS, N_TIME)
    print(f"    {len(u)} visibility points from {N_STATIONS} stations")
    print("[3] Computing forward model (NDFT) ...")
    t0 = time.time()
    vis_noisy, vis_true = forward_observe(gt_image, u, v, FOV_UAS, NOISE_LEVEL)
    print(f"    Forward: {time.time()-t0:.1f}s")
    print("[4] Computing dirty image (FFT gridding) ...")
    dirty = make_dirty_image_fft(vis_noisy, u, v, N_PIX, FOV_UAS)
    print("[5] Computing dirty beam ...")
    beam = make_dirty_beam_fft(u, v, N_PIX, FOV_UAS)
    print(f"[6] Running CLEAN ...")
    t0 = time.time()
    cleaned, components, residual, n_clean = clean_algorithm(
        dirty, beam, gain=CLEAN_GAIN, niter=CLEAN_NITER, threshold=CLEAN_THRESH,
        restore_sigma=RESTORE_SIGMA)
    print(f"    CLEAN: {n_clean} iters in {time.time()-t0:.1f}s")
    cleaned = np.maximum(cleaned, 0)
    print("[7] Computing metrics ...")
    psnr, ssim_val, cc = compute_metrics(gt_image, cleaned)
    print(f"    PSNR = {psnr:.2f} dB")
    print(f"    SSIM = {ssim_val:.4f}")
    print(f"    CC   = {cc:.4f}")
    metrics = {"PSNR": float(psnr), "SSIM": float(ssim_val), "CC": float(cc)}
    print("[8] Saving outputs ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), gt_image)
        np.save(os.path.join(d, "recon_output.npy"), cleaned)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)
    print("[9] Plotting ...")
    plot_results(gt_image, u, v, dirty, cleaned, metrics)
    print(f"\n{'='*60}")
    print("Task 103 COMPLETE")
    print(f"{'='*60}")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `create_crescent_image`, `generate_uv_coverage`, `forward_observe`

In [ ]:
def create_crescent_image(N, fov):
    """Create a crescent-shaped black hole shadow model."""
    x = np.linspace(-fov/2, fov/2, N)
    y = np.linspace(-fov/2, fov/2, N)
    X, Y = np.meshgrid(x, y)
    R = np.sqrt(X**2 + Y**2)
    r_ring = 40.0
    width = 12.0
    ring = np.exp(-0.5 * ((R - r_ring) / (width / 2.35))**2)
    asym = 1.0 + 0.6 * np.cos(np.arctan2(Y, X) - np.pi)
    image = ring * asym
    shadow = 1.0 - np.exp(-0.5 * (R / (r_ring * 0.5))**2)
    image *= shadow
    image = np.maximum(image, 0)
    if image.sum() > 0:
        image /= image.sum()
    return image

def generate_uv_coverage(n_stations, obs_hours, n_time):
    """Generate uv-coverage for an EHT-like VLBI array."""
    stations_lat = np.array([19.8, 37.1, -23.0, 32.7, -67.8, 78.2, 28.3, -30.7])[:n_stations]
    stations_lon = np.array([-155.5, -3.4, -67.8, -109.9, -68.8, 15.5, -16.6, 21.4])[:n_stations]
    lat_rad = np.deg2rad(stations_lat)
    lon_rad = np.deg2rad(stations_lon)
    wavelength_m = 1.3e-3
    earth_radius_m = 6.371e6
    R_lambda = earth_radius_m / wavelength_m
    X_st = R_lambda * np.cos(lat_rad) * np.cos(lon_rad)
    Y_st = R_lambda * np.cos(lat_rad) * np.sin(lon_rad)
    Z_st = R_lambda * np.sin(lat_rad)
    dec = np.deg2rad(12.0)
    ha = np.linspace(-obs_hours/2, obs_hours/2, n_time) * (np.pi / 12.0)
    u_all, v_all = [], []
    for i in range(n_stations):
        for j in range(i+1, n_stations):
            dx = X_st[j] - X_st[i]
            dy = Y_st[j] - Y_st[i]
            dz = Z_st[j] - Z_st[i]
            for h in ha:
                u = np.sin(h)*dx + np.cos(h)*dy
                v = (-np.sin(dec)*np.cos(h)*dx + np.sin(dec)*np.sin(h)*dy + np.cos(dec)*dz)
                u_all.append(u)
                v_all.append(v)
                u_all.append(-u)
                v_all.append(-v)
    u_all = np.array(u_all)
    v_all = np.array(v_all)
    uas_to_rad = np.pi / (180.0 * 3600.0 * 1e6)
    u_all *= uas_to_rad
    v_all *= uas_to_rad
    return u_all, v_all

def forward_observe(image, u, v, fov, noise_level):
    """Compute visibilities via NDFT (vectorized batches)."""
    N = image.shape[0]
    pix_size = fov / N
    x = (np.arange(N) - N/2) * pix_size
    y = (np.arange(N) - N/2) * pix_size
    X, Y = np.meshgrid(x, y)
    x_flat = X.ravel()
    y_flat = Y.ravel()
    img_flat = image.ravel()
    n_vis = len(u)
    vis = np.zeros(n_vis, dtype=complex)
    batch = 200
    for start in range(0, n_vis, batch):
        end = min(start+batch, n_vis)
        phase = -2.0*np.pi*(np.outer(u[start:end], x_flat) + np.outer(v[start:end], y_flat))
        vis[start:end] = np.dot(np.exp(1j*phase), img_flat)
    noise_amp = noise_level * np.abs(vis).max()
    noise = noise_amp * (np.random.randn(n_vis) + 1j*np.random.randn(n_vis)) / np.sqrt(2)
    return vis + noise, vis

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
def compute_metrics(gt, rec):
    gt_n = gt / gt.max() if gt.max() > 0 else gt
    rec_n = rec / rec.max() if rec.max() > 0 else rec
    mse = np.mean((gt_n - rec_n)**2)
    psnr = 10.0*np.log10(1.0/mse) if mse > 1e-15 else 100.0
    dr = max(gt_n.max()-gt_n.min(), rec_n.max()-rec_n.min())
    if dr < 1e-15: dr = 1.0
    ssim_val = ssim(gt_n, rec_n, data_range=dr)
    gz = gt_n - gt_n.mean(); rz = rec_n - rec_n.mean()
    d = np.sqrt(np.sum(gz**2)*np.sum(rz**2))
    cc = np.sum(gz*rz)/d if d > 1e-15 else 0.0
    return float(psnr), float(ssim_val), float(cc)

def plot_results(gt, uv_u, uv_v, dirty, cleaned, metrics):
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    ext = [-FOV_UAS/2, FOV_UAS/2, -FOV_UAS/2, FOV_UAS/2]
    ax = axes[0,0]
    im = ax.imshow(gt, cmap='afmhot', origin='lower', extent=ext)
    ax.set_title("Ground Truth: Black Hole Shadow", fontsize=13)
    ax.set_xlabel("RA offset (μas)"); ax.set_ylabel("Dec offset (μas)")
    plt.colorbar(im, ax=ax, label="Flux density")
    ax = axes[0,1]
    ax.scatter(uv_u, uv_v, s=0.3, alpha=0.3, c='navy')
    ax.set_title(f"UV Coverage ({len(uv_u)} points)", fontsize=13)
    ax.set_xlabel("u (cycles/μas)"); ax.set_ylabel("v (cycles/μas)")
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax = axes[1,0]
    im = ax.imshow(dirty, cmap='afmhot', origin='lower', extent=ext)
    ax.set_title("Dirty Image", fontsize=13)
    ax.set_xlabel("RA offset (μas)"); ax.set_ylabel("Dec offset (μas)")
    plt.colorbar(im, ax=ax, label="Flux density")
    ax = axes[1,1]
    im = ax.imshow(cleaned, cmap='afmhot', origin='lower', extent=ext)
    ax.set_title(f"CLEAN Reconstruction\nPSNR={metrics['PSNR']:.1f}dB, "
                 f"SSIM={metrics['SSIM']:.3f}, CC={metrics['CC']:.3f}", fontsize=12)
    ax.set_xlabel("RA offset (μas)"); ax.set_ylabel("Dec offset (μas)")
    plt.colorbar(im, ax=ax, label="Flux density")
    plt.tight_layout()
    for d in [RESULTS_DIR, ASSETS_DIR]:
        fig.savefig(os.path.join(d, "reconstruction_result.png"), dpi=150, bbox_inches='tight')
        fig.savefig(os.path.join(d, "vis_result.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **ehtim_imaging**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=21.29 dB ← 🔧 修复前: 19.18 dB (Round 1 修复前: 12.98 dB), SSIM=0.4990)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Interferometric Imaging Directly with Closure Phases and Closure Amplitudes
- Repository: https://github.com/achael/eht-imaging
